In [ ]:
# Import statement
import pandas as pd
import numpy as np
import glob
import seaborn as sns
import matplotlib.pyplot as plt
import pingouin as pg
import statsmodels.formula.api as smf

In [ ]:
# Set a few useful variables
preprocdata_dir="preprocessed_data"
preprocdata_file="df_rlwm_clean.csv"

In [ ]:
# Load the preprocessed data
df_rlwm=pd.read_csv(f'{preprocdata_dir}/{preprocdata_file}')
# Drop the Unnamed index created by earlier operations
df_rlwm.drop(columns="Unnamed: 0", inplace=True)
# Keep only the 'learning' phase 
df_rlwm=df_rlwm[df_rlwm['trial_mode']=='learning'].reset_index(drop=True)
# Exclude the first block, which is considered training (it had also twice as many repetitions per stim)
df_rlwm=df_rlwm[df_rlwm['blockId']>0].reset_index(drop=True)


In [ ]:
# Inspect the columns
display(df_rlwm.columns)

In [ ]:
# First of all, let's check if the key prediction of the RLWM task is validated
# However, we realize that we are missing a key component: the repetition count of each stimulus into each block
# This is a key lesson for you: always check before running your whole experiment that you log all the information you need
# to make your analyses!
# Here, fortunately, we are able to recompute this using blockId and stimId information
df_rlwm["stim_rep"] = (
    df_rlwm
    .groupby(["subjid", "blockId", "stimId"])
    .cumcount()
)

In [ ]:
# Now we group by subjid, setSize and stim_rep to reproduce the plot of Collins et al.
df_rlwm_avg=df_rlwm.groupby(by=['subjid', 'setSize', 'stim_rep']).mean(numeric_only=True).reset_index()

In [ ]:
# We make a quick plot using Seaborn, which is essentially a "wrapper" around matplotlib (the core plotting library of Python)
# Seaborn is meant to work with Pandas dataframes, which makes our lives much easier
g=sns.lineplot(data=df_rlwm_avg, x='stim_rep', y='correct', hue='setSize')

In [ ]:
# We make again the same plot using the same color code as Collins et al by defining a custom "palette" (dictionary whose keys match the possible values of setSize)
# We also use a different type of Seaborn plot that gives us exactly the same as in the original article.
collins_palette={
  2.0: (0.2,0.30,0.65),
  3.0: (0.45,0.80,0.86),
  4.0: (0.4,0.75,0.25),
  5.0: (1.0,0.8,0.0)
}
fig,ax = plt.subplots(figsize=(4,4))
sns.pointplot(data=df_rlwm_avg, x='stim_rep', y='correct', hue='setSize', palette=collins_palette, linestyles='--')

In [ ]:
# Now, let see if our result is significant using the statistical package pingouin, which is essentially a wrapper around another more complex library called statsmodels
# Pingouin is meant to work with Pandas dataframes, which again makes our lives much easier!

aov=pg.rm_anova(data=df_rlwm_avg, dv="correct", within=['setSize', 'stim_rep'], subject='subjid')
print(aov.round(3))

# What can you say about the results?
# Anything that bothers you?


In [ ]:
# We will re-run a similar analysis using a mixed-effect model treating setSize and stim_rep as continuous variables
model = smf.mixedlm(
    "correct ~ setSize * stim_rep",
    data=df_rlwm_avg,
    groups=df_rlwm_avg["subjid"]
)

res = model.fit()
print(res.summary())

# Inspect the results below.
# What can you say about them?
# Is there something that bother you?

In [ ]:
# Let's just make sure that it is the random effect covariance that is close to the (0) boundary
print("covariance of the random effects: ", res.cov_re)
# If that's thase, then the warning above is harmless: it simply means the the random intercept associated with our different subjects are not really needed to explain the data
# (subjects are just quite similar to each other)

In [ ]:
# Ok, now as you already realized this version of the RLWM task had a second phase (called test1), performed after a "filler" task lasting about 7 minutes
# In this phase the participants were presented again with all the stimuli multiple times, and they had to give the response they had learn during the learning phase
# However, now, there is now more feedback so they can only use their memory.
# Let's look at this phase

# Load again the preprocessed data into a different dataframe
df_rlwm_test=pd.read_csv(f'{preprocdata_dir}/{preprocdata_file}')
# Drop again the Unnamed index created by earlier operations
df_rlwm_test.drop(columns="Unnamed: 0", inplace=True)
# Now, keep only the 'test1' phase 
df_rlwm_test=df_rlwm_test[df_rlwm_test['trial_mode']=='test1'].reset_index(drop=True)
# Exclude again the first block (probably not needed anyway here)
df_rlwm_test=df_rlwm_test[df_rlwm_test['blockId']>0].reset_index(drop=True)

In [ ]:
# Now, we are not really interested in the repetitions during the test, so we can just average directly without conditioning on this factor
df_rlwm_test_avg=df_rlwm_test.groupby(by=['subjid', 'setSize']).mean(numeric_only=True).reset_index()

In [ ]:
# Let's look at the performance as a function of set size
# We'll just use a bar plot and we will overlap points over it
collins_palette={
  2.0: (0.2,0.30,0.65),
  3.0: (0.45,0.80,0.86),
  4.0: (0.4,0.75,0.25),
  5.0: (1.0,0.8,0.0)
}
fig,ax = plt.subplots(figsize=(4,4))
sns.stripplot(data=df_rlwm_test_avg, y='correct', hue='setSize', palette=collins_palette, ax=ax, alpha=0.5, dodge=True, legend=False, zorder=1)
sns.barplot(data=df_rlwm_test_avg, y='correct', hue='setSize', palette=collins_palette, ax=ax, alpha=0.5, zorder=1)
ax.axhline(y=0.5, linestyle=':', color='k')

# What can you tell "by eye" when looking at this plot?
# Why do you think you are seeing this result?

In [ ]:
# Let's run a statistical analysis to confirm our visual impression
model = smf.mixedlm(
    "correct ~ setSize",
    data=df_rlwm_test_avg,
    groups=df_rlwm_test_avg["subjid"]
)

res = model.fit()
print(res.summary())

In [ ]:
df_rlwm_test_avg

In [ ]:
# Let's digger deeper to see whether overall performance in the learning and test phases are correlated
# To do this, we'll start by merging information from df_rlwm_avg and df_rlwm_test_avg
# However, they have the same column names, so the merge operation would be confusing if we don't do anything about it. Let's first rename the 'correct' columns
df_rlwm_avg.rename(columns={'correct': 'correct_learning'}, inplace=True)
df_rlwm_test_avg.rename(columns={'correct': 'correct_test'}, inplace=True)
# Now, we are ready to merge. We will simply bring the correct_test column of df_rlwm_test_avg into df_rlwm_avg
df_rlwm_avg=df_rlwm_avg.merge(df_rlwm_test_avg[['subjid', 'correct_test']], on="subjid", how="left")
# Actually, df_rlwm_avg still separate data as a function of stim_rep(etition), let's get rid of that
df_rlwm_avg=df_rlwm_avg.groupby(by=['subjid', 'setSize']).mean(numeric_only=True).reset_index()
# Let's look at our new df_rlwm_avg. The correct_test column should appear as the last column
display(df_rlwm_avg)

In [ ]:
# Ok, now, let's plot the correlation between the two types of accuracies
collins_palette={
  2.0: (0.2,0.30,0.65),
  3.0: (0.45,0.80,0.86),
  4.0: (0.4,0.75,0.25),
  5.0: (1.0,0.8,0.0)
}
g=sns.lmplot(data=df_rlwm_avg, x='correct_learning', y='correct_test', hue='setSize', palette=collins_palette, height=4, aspect=1,scatter_kws={"s": 20}, ci=None)

# What can you see?

In [ ]:
# To compute the correlation, let's reduce even further our dataframe to exclude setSize
df_rlwm_avg_nosetsize=df_rlwm_avg.groupby(by=['subjid']).mean(numeric_only=True).reset_index()
# Let plot the overall correlation first
fig, ax= plt.subplots(figsize=(4,4))
g=sns.regplot(data=df_rlwm_avg_nosetsize, x='correct_learning', y='correct_test', color="gray", scatter_kws={"s": 20}, ci=None)

# What can you see? Do you think you can run a standard Pearson correlation test on this data?

In [ ]:
# Let's check if our data is normally distributed
# import the statistical test for normality
from scipy.stats import shapiro

cols = ["correct_learning", "correct_test"]
for col in cols:
    x = df_rlwm_avg_nosetsize[col].dropna()
    stat, p = shapiro(x)
    print(f"{col}: W={stat:.4f}, p={p:.4g}")

# What can you see?

In [ ]:
# Perhaps we should not use a test whose validity depends on a normality assumption...
# We will compare both types of correlation tests using the .corr() methods provided by Pandas
print("Pearson r:", df_rlwm_avg_nosetsize[["correct_learning", "correct_test"]].corr(method="pearson").iloc[0,1])
print("Spearman r:", df_rlwm_avg_nosetsize[["correct_learning", "correct_test"]].corr(method="spearman").iloc[0,1])

# What do you see?

In [ ]:
# To finish, we'll look at the broader correlation structure of our data.
# We don't have any specific hypothesis in mind, we are just curious and we want to look at the relation between our variables all at once 
# This has implications when it comes to reporting such findings in the literature.. Which ones?
# Let's only keep the columns we want for this analysis
df_rlwm_correlation_matrix=df_rlwm_avg_nosetsize[['correct_learning', 'correct_test', 'rt', 'BDI', 'PSS', 'STAI']].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(
    df_rlwm_correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap="vlag",
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={"label": "Pearson r"}
)

plt.title("Correlation matrix")
plt.tight_layout()
plt.show()

# What can you see?